<a href="https://colab.research.google.com/github/simondiange/Breast-cancer-prediction-app/blob/main/Mobilemoney_fraud_detection.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [14]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler, LabelEncoder
from sklearn.metrics import classification_report, confusion_matrix, precision_recall_curve, auc
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv1D, MaxPooling1D, Flatten, Dense, Dropout, Input # Import Input layer
from tensorflow.keras.optimizers import Adam

# --- 1. Data Loading and Preprocessing ---
# Assuming the 'df' DataFrame from your kernel state is the starting point.
# Recreating a similar DataFrame structure for a deployable script.
# In a real scenario, you would load your actual dataset here.
# For demonstration, we use the `df` variable available in the kernel.

# Sample data based on the 'df' in the kernel state
data = {
    'transaction_id': range(100000),
    'sender_id': np.random.randint(1000, 5000, 100000),
    'receiver_id': np.random.randint(1000, 5000, 100000),
    'amount': np.random.rand(100000) * 10000 + 100,
    'transaction_type': np.random.choice(['Payment', 'CashOut', 'Transfer', 'CashIn'], 100000),
    'time_of_day': np.random.randint(0, 24, 100000),
    'day_of_week': np.random.randint(0, 7, 100000),
    'location_code': np.random.randint(0, 20, 100000),
    'is_fraud': np.zeros(100000, dtype=int)
}
df = pd.DataFrame(data)

# Introduce some synthetic fraud cases for demonstration, mirroring your 'fraud_indices'
# There are 150 fraud cases based on your `num_fraud` and `y` variable.
# Let's make it 150 fraud cases for demo.
fraud_indices_demo = np.random.choice(df.index, 150, replace=False)
df.loc[fraud_indices_demo, 'is_fraud'] = 1

print("Original DataFrame head:")
display(df.head())
print(f"Number of fraud transactions in demo data: {df['is_fraud'].sum()}")

# Encode categorical features: 'transaction_type'
# Using LabelEncoder to transform transaction_type into numerical labels.
# We fit this encoder here so it can be used for new transactions later.
label_encoder = LabelEncoder()
df['transaction_type_encoded'] = label_encoder.fit_transform(df['transaction_type'])

# Define features (X) and target (y)
# Features selected based on common practice for fraud detection and your kernel variables
features = ['amount', 'time_of_day', 'day_of_week', 'location_code', 'transaction_type_encoded']
X = df[features]
y = df['is_fraud']

# Split data into training and testing sets
# Stratify by 'is_fraud' to maintain the proportion of fraud cases in both sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)

# Scale numerical features using MinMaxScaler
# We fit the scaler on the training data to prevent data leakage.
scaler = MinMaxScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Reshape data for CNN (samples, timesteps, features)
# For tabular data, we often treat each feature as a timestep of dimension 1.
# This matches the shape of X_train_cnn in your kernel state (e.g., (70000, 5, 1)).
X_train_cnn = X_train_scaled.reshape(X_train_scaled.shape[0], X_train_scaled.shape[1], 1)
X_test_cnn = X_test_scaled.reshape(X_test_scaled.shape[0], X_test_scaled.shape[1], 1)

print("\nShape of X_train_cnn:", X_train_cnn.shape)
print("Shape of X_test_cnn:", X_test_cnn.shape)

# Handle class imbalance
# Using class weights, similar to the `class_weight` variable in your kernel.
neg, pos = np.bincount(y_train)
total = neg + pos
print(f'\nTotal training samples: {total}')
print(f'Number of non-fraudulent samples: {neg} ({100 * neg / total:.2f}% of total)')
print(f'Number of fraudulent samples: {pos} ({100 * pos / total:.2f}% of total)')

initial_bias = np.log([pos / neg])
print(f'Initial bias: {initial_bias}')

class_weight = {0: 1/neg * (total/2.0), 1: 1/pos * (total/2.0)}
print(f'Class weights: {class_weight}')

# --- 2. CNN Model Definition ---
# Define a 1D CNN model suitable for tabular data.
# The architecture can be adapted based on performance.

input_shape = (X_train_cnn.shape[1], 1) # (number_of_features, 1)

model = Sequential([
    Input(shape=input_shape), # Explicitly define the input layer
    Conv1D(filters=32, kernel_size=2, activation='relu'),
    MaxPooling1D(pool_size=1), # Changed pool_size to 1 to avoid dimension collapse
    Dropout(0.3), # Added dropout for regularization
    Conv1D(filters=64, kernel_size=2, activation='relu'),
    MaxPooling1D(pool_size=1), # Changed pool_size to 1 to avoid dimension collapse
    Dropout(0.3), # Added dropout for regularization
    Flatten(),
    Dense(64, activation='relu'),
    Dropout(0.5),
    Dense(1, activation='sigmoid', bias_initializer=tf.keras.initializers.Constant(initial_bias))
])

# Compile the model with appropriate metrics for imbalanced classification
METRICS = [
    tf.keras.metrics.TruePositives(name='tp'),
    tf.keras.metrics.FalsePositives(name='fp'),
    tf.keras.metrics.TrueNegatives(name='tn'),
    tf.keras.metrics.FalseNegatives(name='fn'),
    tf.keras.metrics.BinaryAccuracy(name='accuracy'),
    tf.keras.metrics.Precision(name='precision'),
    tf.keras.metrics.Recall(name='recall'),
    tf.keras.metrics.AUC(name='auc'),
    tf.keras.metrics.AUC(name='pr_auc', curve='PR'), # Precision-Recall AUC
]

model.compile(
    optimizer=Adam(learning_rate=1e-3),
    loss='binary_crossentropy',
    metrics=METRICS
)

model.summary()

# --- 3. Model Training ---
# Train the CNN model with the preprocessed data and class weights.
# Using a small number of epochs for quick demonstration.

EPOCHS = 20
BATCH_SIZE = 256

history = model.fit(
    X_train_cnn, y_train,
    batch_size=BATCH_SIZE,
    epochs=EPOCHS,
    validation_data=(X_test_cnn, y_test),
    class_weight=class_weight,
    verbose=1 # Set to 0 to suppress output during training
)

# --- 4. Model Evaluation ---
# Evaluate the trained model using the test set.

print("\n--- Model Evaluation ---")
results = model.evaluate(X_test_cnn, y_test, batch_size=BATCH_SIZE, verbose=0)
for name, value in zip(model.metrics_names, results):
  print(f"{name}: {value:.4f}")

y_pred_proba = model.predict(X_test_cnn)
y_pred = (y_pred_proba > 0.5).astype(int) # Threshold at 0.5 for binary classification

print("\nConfusion Matrix:")
cm = confusion_matrix(y_test, y_pred)
display(pd.DataFrame(cm, index=['Actual Non-Fraud', 'Actual Fraud'], columns=['Predicted Non-Fraud', 'Predicted Fraud']))

print("\nClassification Report:")
print(classification_report(y_test, y_pred))

# Precision-Recall Curve and AUC (similar to `pr_auc_cnn` in kernel state)
precision_vals, recall_vals, _ = precision_recall_curve(y_test, y_pred_proba)
pr_auc = auc(recall_vals, precision_vals)
print(f"Precision-Recall AUC: {pr_auc:.4f}")

# --- 5. Real-time Fraud Prediction Function (Mimicking Momo Algorithms) ---
# This function simulates how a real-world transaction would be processed for fraud detection.

def predict_momo_fraud(
    amount: float,
    time_of_day: int, # 0-23
    day_of_week: int, # 0-6 (Monday=0, Sunday=6)
    location_code: int, # e.g., location ID
    transaction_type_str: str # e.g., 'Payment', 'CashOut', 'Transfer', 'CashIn'
) -> dict:
    """
    Predicts if a Mobile Money transaction is fraudulent.

    Args:
        amount (float): The transaction amount.
        time_of_day (int): Hour of the day (0-23).
        day_of_week (int): Day of the week (0=Monday, 6=Sunday).
        location_code (int): A numerical code for the transaction location.
        transaction_type_str (str): The type of transaction (e.g., 'Payment', 'CashOut', 'Transfer', 'CashIn').

    Returns:
        dict: A dictionary containing the prediction ('Fraudulent' or 'Legitimate')
              and the fraud probability score.
    """

    # Create a DataFrame for the new transaction
    new_transaction_df = pd.DataFrame([{
        'amount': amount,
        'time_of_day': time_of_day,
        'day_of_week': day_of_week,
        'location_code': location_code,
        'transaction_type': transaction_type_str
    }])

    # Apply the same preprocessing steps as the training data
    # 1. Encode transaction type
    try:
        new_transaction_df['transaction_type_encoded'] = label_encoder.transform(new_transaction_df['transaction_type'])
    except ValueError:
        return {
            'prediction': 'Error',
            'message': f"Unknown transaction type: {transaction_type_str}. Valid types are {list(label_encoder.classes_)}"
        }

    # Select features based on the `features` list used for training
    transaction_features = new_transaction_df[features]

    # 2. Scale numerical features
    transaction_scaled = scaler.transform(transaction_features)

    # 3. Reshape for CNN
    transaction_cnn = transaction_scaled.reshape(1, transaction_scaled.shape[1], 1)

    # Make prediction
    fraud_probability = model.predict(transaction_cnn)[0][0]

    # Determine if it's fraud based on a threshold (e.g., 0.5 or a tuned threshold)
    prediction_label = 'Fraudulent' if fraud_probability > 0.5 else 'Legitimate'

    return {
        'prediction': prediction_label,
        'fraud_probability': float(fraud_probability)
    }

# --- Demo of the Real-time Prediction Function ---
print("\n--- Demo of Real-time Prediction ---")

# Example 1: A legitimate-looking transaction
legitimate_transaction = predict_momo_fraud(
    amount=500.0,
    time_of_day=10,
    day_of_week=2, # Wednesday
    location_code=5,
    transaction_type_str='Payment'
)
print("Legitimate transaction prediction:", legitimate_transaction)

# Example 2: A potentially fraudulent transaction (e.g., high amount, unusual time/type)
# Note: This is synthetic; a real fraudulent transaction might have different characteristics.
fraudulent_transaction = predict_momo_fraud(
    amount=9500.0,
    time_of_day=2,
    day_of_week=6, # Sunday night/early morning
    location_code=18,
    transaction_type_str='CashOut'
)
print("Potentially fraudulent transaction prediction:", fraudulent_transaction)

# Example 3: Another legitimate transaction
legitimate_transaction_2 = predict_momo_fraud(
    amount=1500.0,
    time_of_day=15,
    day_of_week=4, # Friday
    location_code=1,
    transaction_type_str='Transfer'
)
print("Another legitimate transaction prediction:", legitimate_transaction_2)

# Example 4: Invalid transaction type
invalid_transaction = predict_momo_fraud(
    amount=100.0,
    time_of_day=12,
    day_of_week=1,
    location_code=3,
    transaction_type_str='UnknownType'
)
print("Invalid transaction type prediction:", invalid_transaction)


Original DataFrame head:


,transaction_id,sender_id,receiver_id,amount,transaction_type,time_of_day,day_of_week,location_code,is_fraud
0,0,1803,1240,8012.913857,CashOut,17,3,11,0
1,1,2298,4283,9467.401708,CashOut,8,0,14,0
2,2,3918,4591,9206.147433,CashIn,14,1,7,0
3,3,4280,2025,1499.297634,CashOut,4,1,15,0
4,4,3990,2508,6089.747612,CashOut,19,0,2,0


Number of fraud transactions in demo data: 150

Shape of X_train_cnn: (70000, 5, 1)
Shape of X_test_cnn: (30000, 5, 1)

Total training samples: 70000
Number of non-fraudulent samples: 69895 (99.85% of total)
Number of fraudulent samples: 105 (0.15% of total)
Initial bias: [-6.50078904]
Class weights: {0: np.float64(0.5007511266900351), 1: np.float64(333.33333333333337)}


Model: "sequential_4"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv1d_7 (Conv1D)               │ (None, 4, 32)          │            96 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d_7 (MaxPooling1D)  │ (None, 4, 32)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_10 (Dropout)            │ (None, 4, 32)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_8 (Conv1D)               │ (None, 3, 64)          │         4,160 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d_8 (MaxPooling1D)  │ (None, 3, 64)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_11 (Dropout)            │ (None, 3, 64)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_4 (Flatten)             │ (None, 192)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_8 (Dense)                 │ (None, 64)             │        12,352 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_12 (Dropout)            │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_9 (Dense)                 │ (None, 1)              │            65 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

...

 Trainable params: 16,673 (65.13 KB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/20
274/274 ━━━━━━━━━━━━━━━━━━━━ 7s 15ms/step - accuracy: 0.9021 - auc: 0.5024 - fn: 99.0000 - fp: 6755.0000 - loss: 2.4865 - pr_auc: 0.0014 - precision: 8.8744e-04 - recall: 0.0571 - tn: 63140.0000 - tp: 6.0000 - val_accuracy: 0.3062 - val_auc: 0.4796 - val_fn: 11.0000 - val_fp: 20803.0000 - val_loss: 1.2365 - val_pr_auc: 0.0014 - val_precision: 0.0016 - val_recall: 0.7556 - val_tn: 9152.0000 - val_tp: 34.0000
Epoch 2/20
274/274 ━━━━━━━━━━━━━━━━━━━━ 3s 12ms/step - accuracy: 0.6676 - auc: 0.5136 - fn: 62.0000 - fp: 23209.0000 - loss: 1.1441 - pr_auc: 0.0015 - precision: 0.0018 - recall: 0.4095 - tn: 46686.0000 - tp: 43.0000 - val_accuracy: 0.4706 - val_auc: 0.4818 - val_fn: 23.0000 - val_fp: 15858.0000 - val_loss: 0.7850 - val_pr_auc: 0.0014 - val_precision: 0.0014 - val_recall: 0.4889 - val_tn: 14097.0000 - val_tp: 22.0000
Epoch 3/20
274/274 ━━━━━━━━━━━━━━━━━━━━ 4s 15ms/step - accuracy: 0.6255 - auc: 0.5000 - fn: 69.0000 - fp: 26145.0000 - loss: 1.0957 - pr_auc: 0.0015 - precis

,Predicted Non-Fraud,Predicted Fraud
Actual Non-Fraud,443,29512
Actual Fraud,0,45



Classification Report:
              precision    recall  f1-score   support

           0       1.00      0.01      0.03     29955
           1       0.00      1.00      0.00        45

    accuracy                           0.02     30000
   macro avg       0.50      0.51      0.02     30000
weighted avg       1.00      0.02      0.03     30000

Precision-Recall AUC: 0.0015

--- Demo of Real-time Prediction ---
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 58ms/step
Legitimate transaction prediction: {'prediction': 'Fraudulent', 'fraud_probability': 0.613240659236908}
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 75ms/step
Potentially fraudulent transaction prediction: {'prediction': 'Fraudulent', 'fraud_probability': 0.5353976488113403}
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 143ms/step
Another legitimate transaction prediction: {'prediction': 'Fraudulent', 'fraud_probability': 0.6020621657371521}
Invalid transaction type prediction: {'prediction': 'Error', 'message': "Unknown transaction type: UnknownType. Valid types are ['Cas